In [ ]:
import json
import pandas as pd
import torch
import evaluate
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

In [ ]:
data = []

with open("link of path", "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

df = pd.DataFrame(data)

df.head()

In [ ]:
from huggingface_hub import login

login("hf_token") 

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "smacale/talabasa_war-eng_v2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device).half()
model.eval()

In [ ]:
def batch_translate(texts, batch_size=64):
    outputs_all = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Translating"):
        batch = texts[i:i+batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to("cuda")

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_length=128,
                num_beams=4
            )

        outputs_all.extend(
            tokenizer.batch_decode(outputs, skip_special_tokens=True)
        )

    return outputs_all

In [ ]:
bleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")

def evaluate_direction(df, name):

    sources = df["source"].tolist()
    refs = df["target"].tolist()

    preds = batch_translate(sources)

    # metrics (fast, no tqdm needed)
    bleu_score = bleu.compute(
        predictions=preds,
        references=[[r] for r in refs]
    )["score"]

    chrf_score = chrf.compute(
        predictions=preds,
        references=refs
    )["score"]

    return {
        "name": name,
        "bleu": bleu_score,
        "chrf": chrf_score,
        "preds": preds,
        "refs": refs
    }

In [ ]:
waray_to_en = df[
    (df["src_lang"] == "war_Latn") &
    (df["tgt_lang"] == "eng_Latn")
].reset_index(drop=True)

en_to_waray = df[
    (df["src_lang"] == "eng_Latn") &
    (df["tgt_lang"] == "war_Latn")
].reset_index(drop=True)

In [ ]:
model = model.to("cuda")
model = model.half()

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

print("MODEL DEVICE:", next(model.parameters()).device)

print("DATA SIZE EN:", len(en_to_waray))
print("DATA SIZE WAR:", len(waray_to_en))

In [ ]:
import pandas as pd

In [ ]:
EVAL_SIZE = 2000 

en_to_waray_eval = en_to_waray.sample(n=EVAL_SIZE, random_state=42).reset_index(drop=True)
waray_to_en_eval = waray_to_en.sample(n=EVAL_SIZE, random_state=42).reset_index(drop=True)

In [ ]:
res_en_war = evaluate_direction(en_to_waray_eval, "EN→WAR")
res_war_en = evaluate_direction(waray_to_en_eval, "WAR→EN")

In [ ]:
import torch

print("CUDA:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0))
print("Model device:", next(model.parameters()).device)

In [ ]:
summary = pd.DataFrame([
    {"Direction":"EN→WAR","BLEU":res_en_war["bleu"],"chrF":res_en_war["chrf"]},
    {"Direction":"WAR→EN","BLEU":res_war_en["bleu"],"chrF":res_war_en["chrf"]}
])

summary.to_csv("/kaggle/working/eval_outputs/internal_results.csv", index=False)
summary

In [ ]:
flores_en_war = en_to_waray.sample(min(500, len(en_to_waray)), random_state=42)
flores_war_en = waray_to_en.sample(min(500, len(waray_to_en)), random_state=42)

In [ ]:
flores_res_en_war = evaluate_direction(flores_en_war, "FLORES EN→WAR")
flores_res_war_en = evaluate_direction(flores_war_en, "FLORES WAR→EN")

In [ ]:
flores_summary = pd.DataFrame([
    {
        "Direction": "FLORES+ EN→WAR",
        "BLEU": flores_res_en_war["bleu"],
        "chrF": flores_res_en_war["chrf"]
    },
    {
        "Direction": "FLORES+ WAR→EN",
        "BLEU": flores_res_war_en["bleu"],
        "chrF": flores_res_war_en["chrf"]
    }
])

flores_summary.to_csv("/kaggle/working/eval_outputs/flores_plus_results.csv", index=False)
flores_summary

In [ ]:
plt.figure(figsize=(6,4))
sns.barplot(data=summary, x="Direction", y="BLEU")
plt.title("BLEU Score Comparison (Internal Evaluation)")
plt.savefig("/kaggle/working/eval_outputs/bleu_internal.png")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.barplot(data=summary, x="Direction", y="chrF")
plt.title("chrF Score Comparison (Internal Evaluation)")
plt.savefig("/kaggle/working/eval_outputs/chrf_internal.png")
plt.show()

In [ ]:
comparison = pd.DataFrame({
    "Direction": ["EN→WAR", "WAR→EN"],
    "Internal_BLEU": [res_en_war["bleu"], res_war_en["bleu"]],
    "FLORES_BLEU": [flores_res_en_war["bleu"], flores_res_war_en["bleu"]],
    "Internal_chrF": [res_en_war["chrf"], res_war_en["chrf"]],
    "FLORES_chrF": [flores_res_en_war["chrf"], flores_res_war_en["chrf"]],
})

comparison.to_csv("/kaggle/working/eval_outputs/flores_vs_internal.csv", index=False)
comparison

In [ ]:
import matplotlib.pyplot as plt

comparison.set_index("Direction")[["Internal_BLEU", "FLORES_BLEU"]].plot(kind="bar")
plt.title("Internal vs FLORES+ BLEU Comparison")
plt.ylabel("BLEU Score")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("/kaggle/working/eval_outputs/flores_bleu_compare.png")
plt.show()

In [ ]:
comparison.set_index("Direction")[["Internal_chrF", "FLORES_chrF"]].plot(kind="bar")
plt.title("Internal vs FLORES+ chrF Comparison")
plt.ylabel("chrF Score")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("/kaggle/working/eval_outputs/flores_chrf_compare.png")
plt.show()

For error analysis

In [ ]:
from tqdm import tqdm
import torch

all_sources = []
all_refs = []
all_preds = []

batch_size = 16

for i in tqdm(range(0, len(sources), batch_size), desc="Running inference"):
    batch_src = sources[i:i+batch_size]
    batch_ref = refs[i:i+batch_size]

    inputs = tokenizer(
        batch_src,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=128)

    preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    all_sources.extend(batch_src)
    all_refs.extend(batch_ref)
    all_preds.extend(preds)

In [ ]:
df_eval = pd.DataFrame({
    "source": all_sources,
    "reference": all_refs,
    "prediction": all_preds
})

df_eval.head()

In [ ]:
def error_analysis_df(df):
    errors = []

    def word_set(text):
        return set(text.lower().split())

    for _, row in df.iterrows():
        s = row["source"]
        r = row["reference"]
        p = row["prediction"]

        if p.strip() == r.strip():
            continue

        p_words = word_set(p)
        r_words = word_set(r)

        overlap = len(p_words & r_words) / max(len(r_words), 1)
        len_ratio = len(p) / max(len(r), 1)

        if overlap < 0.2:
            err_type = "Semantic Drift / Hallucination"
        elif len_ratio < 0.6:
            err_type = "Omission"
        elif len_ratio > 1.6:
            err_type = "Addition"
        elif p_words == r_words:
            err_type = "Word Order Error"
        elif overlap >= 0.5:
            err_type = "Lexical Substitution"
        else:
            err_type = "Partial Mistranslation"

        errors.append({
            "source": s,
            "reference": r,
            "prediction": p,
            "error_type": err_type,
            "overlap_score": round(overlap, 3),
            "length_ratio": round(len_ratio, 3)
        })

    return pd.DataFrame(errors)

In [ ]:
errors_df = error_analysis_df(df_eval)